# Proyecto 3 — Call Center Analytics
**Notebook 01 · Exploración inicial y validación de calidad del dataset WFO**

**Recorrido**
1. Objetivo, fuente y alcance
2. Exploración inicial
3. Validación de fechas, volúmenes y tiempos
4. Revisión de canal, Tier y personal
5. Granularidad y claves candidatas
6. Cobertura temporal
7. Resumen de hallazgos y decisiones
8. Preparación para SQL Server

---

## 1. Objetivo, fuente y alcance

* **Objetivo:** verificar la estructura y la calidad del dataset antes de modelarlo en SQL Server y Power BI, y documentar qué problemas tiene y cómo se tratan.
* **Fuente:** archivo `call_center_operations_wfo_dataset.csv` (10,000 filas × 30 columnas), con un registro por fecha, hora, sede y cola. *https://www.kaggle.com/datasets/komalkomal7/call-center-workforce-dataset-wfo-aht-kpi*
* **Naturaleza de los datos:** sintéticos. No incluyen identificador de agente, solo conteos agregados de personal.
* **Alcance:** exploración y validación. No se eliminan filas ni se inventan valores: los problemas se señalan con columnas de validación sobre una copia de trabajo. La carga a SQL Server queda fuera de este notebook.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Rutas relativas: ejecutar desde la raíz del proyecto o desde notebooks/.
from pathlib import Path

opciones = [
    Path("data/raw/call_center_operations_wfo_dataset.csv"),
    Path("../data/raw/call_center_operations_wfo_dataset.csv"),
    Path("call_center_operations_wfo_dataset.csv"),
]
ruta = next((p for p in opciones if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError(
        "No se encontró el CSV. Abre el proyecto desde su carpeta raíz "
        "o ejecuta el notebook dentro de notebooks/."
    )
df = pd.read_csv(ruta)
df.head()

,Date,Weekday,Week_Number,Month,Year,Hour,Country,Region,Site,Queue_Name,...,aHT_Seconds,handle_Time_Total,Wait_Time_Avg,shrinkage_Percent,required_FTE,agent_Count,utilization_percent,Active_Agents,Service_Level_Percent,Occupancy_Percent
0,29-09-2023,Friday,39,September,2023,3,Bulgaria,EMEA,Bulgaria Hub,INBOUND_VOICE_TIER1,...,355,171820,5.772358,28,5,6,71,4,86,89
1,27-10-2022,Thursday,43,October,2022,16,Malaysia,APAC,Kuala Lumpur,TECH_SUPPORT_EMAIL_T3,...,613,53944,78.900990,31,1,1,75,1,82,60
2,16-10-2022,Sunday,41,October,2022,15,Kenya,EMEA,Kenya Hub,EMEA_CHAT_SUPPORT_T1,...,272,17952,44.759494,18,0,0,90,0,73,71
3,23-06-2021,Wednesday,25,June,2021,12,New Zealand,APAC,New Zealand Hub,APAC_EMAIL_SUPPORT_T3,...,459,99144,6.287671,32,3,4,91,4,82,73
4,24-10-2021,Sunday,42,October,2021,18,Bangladesh,APAC,Bangladesh Hub,APAC_CORE_EMAIL_TIER2,...,867,384948,44.461538,27,13,17,71,12,77,73


In [4]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.info()

Filas: 10000
Columnas: 30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   10000 non-null  object 
 1   Weekday                10000 non-null  object 
 2   Week_Number            10000 non-null  int64  
 3   Month                  10000 non-null  object 
 4   Year                   10000 non-null  int64  
 5   Hour                   10000 non-null  int64  
 6   Country                10000 non-null  object 
 7   Region                 10000 non-null  object 
 8   Site                   10000 non-null  object 
 9   Queue_Name             10000 non-null  object 
 10  Channel                10000 non-null  object 
 11  All_Units_Filter       10000 non-null  object 
 12  Queue_Units_Filter     10000 non-null  object 
 13  ASU_Tier_Filter        10000 non-null  object 
 14  Business_Unit          10000 

In [5]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique()
})

resumen

,tipo,nulos,valores_distintos
Date,object,0,1885
Weekday,object,0,7
Week_Number,int64,0,53
Month,object,0,12
Year,int64,0,6
Hour,int64,0,24
Country,object,0,53
Region,object,0,3
Site,object,0,58
Queue_Name,object,0,20


In [6]:
print("Filas completamente duplicadas:", df.duplicated().sum())

for columna in ["Region", "Channel", "Business_Unit"]:
    print(f"\nDistribución de {columna}:")
    print(df[columna].value_counts(dropna=False))

Filas completamente duplicadas: 0

Distribución de Region:
Region
EMEA    5613
APAC    3343
AMER    1044
Name: count, dtype: int64

Distribución de Channel:
Channel
Voice    3382
Email    3325
Chat     3293
Name: count, dtype: int64

Distribución de Business_Unit:
Business_Unit
E-commerce    2081
Healthcare    2024
Retail        1978
Telecom       1963
Banking       1954
Name: count, dtype: int64


In [7]:
df.iloc[0]

Date                              29-09-2023
Weekday                               Friday
Week_Number                               39
Month                              September
Year                                    2023
Hour                                       3
Country                             Bulgaria
Region                                  EMEA
Site                            Bulgaria Hub
Queue_Name               INBOUND_VOICE_TIER1
Channel                                 Chat
All_Units_Filter                    Consumer
Queue_Units_Filter                   Inbound
ASU_Tier_Filter                       Tier 1
Business_Unit                        Banking
planned_Volume                           464
offered_Volume                           492
handled_Volume                           484
abandoned_Volume                           8
backlog_Volume                             8
aHT_Seconds                              355
handle_Time_Total                     171820
Wait_Time_

In [8]:
columnas = [
    "Date", "Hour", "Site", "Queue_Name", "Channel",
    "offered_Volume", "handled_Volume", "abandoned_Volume",
    "agent_Count"
]

df.loc[0, columnas]

Date                         29-09-2023
Hour                                  3
Site                       Bulgaria Hub
Queue_Name          INBOUND_VOICE_TIER1
Channel                            Chat
offered_Volume                      492
handled_Volume                      484
abandoned_Volume                      8
agent_Count                           6
Name: 0, dtype: object

In [9]:
df_trabajo = df.copy()

In [10]:
df_trabajo["Date"] = pd.to_datetime(
    df_trabajo["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [11]:
fechas_invalidas = df_trabajo["Date"].isna()

print("Tipo de dato:", df_trabajo["Date"].dtype)
print("Fechas inválidas:", fechas_invalidas.sum())
print("Fecha mínima:", df_trabajo["Date"].min())
print("Fecha máxima:", df_trabajo["Date"].max())

# Ver los valores originales que no pudieron convertirse
df.loc[fechas_invalidas, ["Date"]]

Tipo de dato: datetime64[ns]
Fechas inválidas: 0
Fecha mínima: 2021-01-01 00:00:00
Fecha máxima: 2026-12-03 00:00:00


,Date


In [12]:
diferencia_volumen = (
    df_trabajo["offered_Volume"]
    - df_trabajo["handled_Volume"]
    - df_trabajo["abandoned_Volume"]
)

volumen_inconsistente = diferencia_volumen.ne(0)

print("Filas revisadas:", len(df_trabajo))
print("Filas que cuadran:", (~volumen_inconsistente).sum())
print("Filas con diferencias:", volumen_inconsistente.sum())

Filas revisadas: 10000
Filas que cuadran: 10000
Filas con diferencias: 0


In [13]:
columnas_revision = [
    "Date", "Hour", "Site", "Queue_Name", "Channel",
    "offered_Volume", "handled_Volume", "abandoned_Volume"
]

df_trabajo.loc[
    volumen_inconsistente,
    columnas_revision
].head(10)

,Date,Hour,Site,Queue_Name,Channel,offered_Volume,handled_Volume,abandoned_Volume


In [14]:
columnas_volumen = [
    "planned_Volume",
    "offered_Volume",
    "handled_Volume",
    "abandoned_Volume",
    "backlog_Volume"
]

negativos_por_columna = (
    df_trabajo[columnas_volumen] < 0
).sum()

print(negativos_por_columna)

planned_Volume      0
offered_Volume      0
handled_Volume      0
abandoned_Volume    0
backlog_Volume      0
dtype: int64


In [15]:
tiempo_total_esperado = (
    df_trabajo["handled_Volume"]
    * df_trabajo["aHT_Seconds"]
)

tiempo_consistente = np.isclose(
    df_trabajo["handle_Time_Total"],
    tiempo_total_esperado,
    rtol=0,
    atol=0.01
)

print("Filas revisadas:", len(df_trabajo))
print("Filas que cuadran:", tiempo_consistente.sum())
print("Filas con diferencias:", (~tiempo_consistente).sum())

Filas revisadas: 10000
Filas que cuadran: 10000
Filas con diferencias: 0


In [16]:
df_trabajo.loc[
    ~tiempo_consistente,
    [
        "handled_Volume",
        "aHT_Seconds",
        "handle_Time_Total"
    ]
].head(10)

,handled_Volume,aHT_Seconds,handle_Time_Total


In [17]:
columnas_tiempo = [
    "aHT_Seconds",
    "handle_Time_Total",
    "Wait_Time_Avg"
]

print((df_trabajo[columnas_tiempo] < 0).sum())

aHT_Seconds          0
handle_Time_Total    0
Wait_Time_Avg        0
dtype: int64


In [18]:
columnas_porcentaje = [
    "shrinkage_Percent",
    "utilization_percent",
    "Service_Level_Percent",
    "Occupancy_Percent"
]

porcentajes_fuera_rango = (
    (df_trabajo[columnas_porcentaje] < 0)
    | (df_trabajo[columnas_porcentaje] > 100)
)

print(porcentajes_fuera_rango.sum())

shrinkage_Percent        0
utilization_percent      0
Service_Level_Percent    0
Occupancy_Percent        0
dtype: int64


In [19]:
resumen_porcentajes = (
    df_trabajo[columnas_porcentaje]
    .agg(["min", "max"])
    .T
)

resumen_porcentajes

,min,max
shrinkage_Percent,15,35
utilization_percent,65,95
Service_Level_Percent,70,95
Occupancy_Percent,60,90


In [20]:
canal_segun_nombre = (
    df_trabajo["Queue_Name"]
    .str.upper()
    .str.extract(r"(VOICE|EMAIL|CHAT)", expand=False)
)

canal_declarado = (
    df_trabajo["Channel"]
    .str.strip()
    .str.upper()
)

In [21]:
canal_identificable = canal_segun_nombre.notna()

conflicto_canal = (
    canal_identificable
    & canal_segun_nombre.ne(canal_declarado)
)

print("Filas revisadas:", len(df_trabajo))
print("Canal identificable en el nombre:", canal_identificable.sum())
print("Sin canal identificable:", (~canal_identificable).sum())
print("Diferencias entre nombre y canal:", conflicto_canal.sum())

Filas revisadas: 10000
Canal identificable en el nombre: 9014
Sin canal identificable: 986
Diferencias entre nombre y canal: 5980


In [22]:
df_trabajo.loc[
    conflicto_canal,
    ["Queue_Name", "Channel"]
].head(10)

,Queue_Name,Channel
0,INBOUND_VOICE_TIER1,Chat
1,TECH_SUPPORT_EMAIL_T3,Chat
2,EMEA_CHAT_SUPPORT_T1,Email
3,APAC_EMAIL_SUPPORT_T3,Voice
4,APAC_CORE_EMAIL_TIER2,Chat
6,APAC_CORE_EMAIL_TIER2,Voice
8,APAC_CORE_EMAIL_T2,Chat
10,APAC_CHAT_SUPPORT_T1,Voice
11,CUSTOMER_SUPPORT_CHAT_T1,Email
16,AMER_RETAIL_VOICE_T2,Email


In [23]:
df_trabajo["validacion_canal"] = "No verificable"

df_trabajo.loc[
    canal_identificable & ~conflicto_canal,
    "validacion_canal"
] = "Coincide"

df_trabajo.loc[
    conflicto_canal,
    "validacion_canal"
] = "Conflicto"

print(df_trabajo["validacion_canal"].value_counts())

validacion_canal
Conflicto         5980
Coincide          3034
No verificable     986
Name: count, dtype: int64


* Hallazgo: 5,980 registros presentan diferencias entre el canal declarado y el identificado en el nombre de la cola.
* Casos no verificables: 986 registros no contienen VOICE, EMAIL o CHAT en el nombre.
* Tratamiento provisional: conservar ambos campos y añadir una columna de validación.
* Limitación: no se ha determinado cuál de los dos campos es correcto; los análisis conjuntos por cola y canal requieren cautela.

In [24]:
print(
    df_trabajo.loc[
        ~canal_identificable,
        "Queue_Name"
    ].value_counts()
)

Queue_Name
OUTBOUND_SALES_T2    986
Name: count, dtype: int64


In [25]:
canales_outbound = df_trabajo.loc[
    df_trabajo["Queue_Name"].eq("OUTBOUND_SALES_T2"),
    "Channel"
].value_counts(dropna=False)

print(canales_outbound)

Channel
Voice    348
Chat     325
Email    313
Name: count, dtype: int64


* La cola OUTBOUND_SALES_T2 contiene 986 registros distribuidos entre Voice, Chat y Email. Como su nombre no especifica un canal, no se utiliza para validar ni reemplazar Channel. Los registros se conservan con la clasificación “No verificable”.

In [26]:
columnas_personal = [
    "required_FTE",
    "agent_Count",
    "Active_Agents"
]

print((df_trabajo[columnas_personal] < 0).sum())

required_FTE     0
agent_Count      0
Active_Agents    0
dtype: int64


In [27]:
activos_superan_total = (
    df_trabajo["Active_Agents"]
    > df_trabajo["agent_Count"]
)

atencion_sin_agentes = (
    df_trabajo["agent_Count"].eq(0)
    & df_trabajo["handled_Volume"].gt(0)
)

print(
    "Agentes activos mayores que el total:",
    activos_superan_total.sum()
)

print(
    "Contactos atendidos con cero agentes:",
    atencion_sin_agentes.sum()
)

Agentes activos mayores que el total: 0
Contactos atendidos con cero agentes: 633


* 633 registros, equivalentes al 6.33 % del dataset, tienen contactos atendidos con cero agentes declarados.

In [28]:
df_trabajo.loc[
    activos_superan_total | atencion_sin_agentes,
    [
        "Date", "Hour", "Queue_Name", "Channel",
        "handled_Volume", "required_FTE",
        "agent_Count", "Active_Agents"
    ]
].head(10)

,Date,Hour,Queue_Name,Channel,handled_Volume,required_FTE,agent_Count,Active_Agents
2,2022-10-16,15,EMEA_CHAT_SUPPORT_T1,Email,66,0,0,0
8,2024-04-09,10,APAC_CORE_EMAIL_T2,Chat,71,0,0,0
11,2021-07-17,23,CUSTOMER_SUPPORT_CHAT_T1,Email,107,0,0,0
31,2022-06-03,15,APAC_CHAT_SUPPORT_T1,Chat,56,0,0,0
43,2024-06-12,5,APAC_CORE_EMAIL_T2,Voice,134,0,0,0
51,2022-09-02,16,INBOUND_VOICE_TIER1,Email,97,0,0,0
61,2025-05-26,4,AMER_CORE_EMAIL_T2,Email,39,0,0,0
63,2023-04-22,18,EMEA_EMAIL_SUPPORT_T3,Chat,48,0,0,0
72,2023-09-26,21,TECH_SUPPORT_EMAIL_T3,Voice,51,0,0,0
87,2024-02-20,3,APAC_RETAIL_VOICE_T2,Email,40,0,0,0


In [29]:
df_trabajo["alerta_atencion_sin_agentes"] = (
    atencion_sin_agentes
)

print(
    df_trabajo["alerta_atencion_sin_agentes"]
    .value_counts()
)

alerta_atencion_sin_agentes
False    9367
True      633
Name: count, dtype: int64


In [30]:
resumen_alerta_personal = df_trabajo.loc[
    atencion_sin_agentes,
    [
        "handled_Volume",
        "required_FTE",
        "agent_Count",
        "Active_Agents"
    ]
].agg(["min", "max"])

resumen_alerta_personal

,handled_Volume,required_FTE,agent_Count,Active_Agents
min,3,0,0,0
max,149,0,0,0


In [31]:
backlog_igual_abandono = (
    df_trabajo["backlog_Volume"]
    .eq(df_trabajo["abandoned_Volume"])
)

print("Filas revisadas:", len(df_trabajo))
print("Valores iguales:", backlog_igual_abandono.sum())
print("Valores diferentes:", (~backlog_igual_abandono).sum())

Filas revisadas: 10000
Valores iguales: 10000
Valores diferentes: 0


In [32]:
df_trabajo.loc[
    ~backlog_igual_abandono,
    [
        "Date", "Channel", "Queue_Name",
        "backlog_Volume", "abandoned_Volume"
    ]
].head(10)

,Date,Channel,Queue_Name,backlog_Volume,abandoned_Volume


In [33]:
clave_candidata = [
    "Date",
    "Hour",
    "Site",
    "Queue_Name",
    "Channel"
]

filas_clave_repetida = df_trabajo.duplicated(
    subset=clave_candidata,
    keep=False
)

print("Filas revisadas:", len(df_trabajo))

print(
    "Combinaciones distintas:",
    df_trabajo[clave_candidata].drop_duplicates().shape[0]
)

print(
    "Filas que comparten la combinación:",
    filas_clave_repetida.sum()
)

Filas revisadas: 10000
Combinaciones distintas: 9999
Filas que comparten la combinación: 2


In [34]:
df_trabajo.loc[
    filas_clave_repetida,
    clave_candidata + [
        "Business_Unit",
        "All_Units_Filter",
        "ASU_Tier_Filter",
        "offered_Volume"
    ]
].sort_values(clave_candidata).head(10)

,Date,Hour,Site,Queue_Name,Channel,Business_Unit,All_Units_Filter,ASU_Tier_Filter,offered_Volume
5339,2021-03-18,20,Luxembourg Hub,EMEA_CORE_EMAIL_T2,Email,Telecom,Premium,Tier 3,410
8991,2021-03-18,20,Luxembourg Hub,EMEA_CORE_EMAIL_T2,Email,E-commerce,Enterprise,Tier 2,263


In [35]:
clave_ampliada = [
    "Date",
    "Hour",
    "Country",
    "Region",
    "Site",
    "Queue_Name",
    "Channel",
    "Business_Unit",
    "All_Units_Filter",
    "Queue_Units_Filter",
    "ASU_Tier_Filter"
]

filas_clave_ampliada_repetida = df_trabajo.duplicated(
    subset=clave_ampliada,
    keep=False
)

print("Filas revisadas:", len(df_trabajo))

print(
    "Combinaciones distintas:",
    df_trabajo[clave_ampliada].drop_duplicates().shape[0]
)

print(
    "Filas que comparten la combinación:",
    filas_clave_ampliada_repetida.sum()
)

Filas revisadas: 10000
Combinaciones distintas: 10000
Filas que comparten la combinación: 0


* **Hallazgo:** - La combinación inicial de fecha, hora, sede, cola y canal no era única: dos registros compartían esos valores, pero pertenecían a segmentos diferentes. Al incluir los demás atributos descriptivos, se obtuvieron 10,000 combinaciones únicas. Esta combinación se considera una clave candidata para el archivo analizado.

In [36]:
tier_segun_nombre = (
    df_trabajo["Queue_Name"]
    .str.strip()
    .str.upper()
    .str.extract(r"(?:TIER|T)_?(\d+)$", expand=False)
)

tier_declarado = (
    df_trabajo["ASU_Tier_Filter"]
    .str.strip()
    .str.upper()
    .str.extract(r"^TIER\s*(\d+)$", expand=False)
)

In [37]:
tier_comparable = (
    tier_segun_nombre.notna()
    & tier_declarado.notna()
)

conflicto_tier = (
    tier_comparable
    & tier_segun_nombre.ne(tier_declarado)
)

print("Filas comparables:", tier_comparable.sum())
print("Filas no verificables:", (~tier_comparable).sum())
print("Coincidencias:", (tier_comparable & ~conflicto_tier).sum())
print("Diferencias:", conflicto_tier.sum())

Filas comparables: 10000
Filas no verificables: 0
Coincidencias: 3342
Diferencias: 6658


In [38]:
df_trabajo.loc[
    conflicto_tier,
    ["Queue_Name", "ASU_Tier_Filter"]
].head(10)

,Queue_Name,ASU_Tier_Filter
1,TECH_SUPPORT_EMAIL_T3,Tier 2
3,APAC_EMAIL_SUPPORT_T3,Tier 2
4,APAC_CORE_EMAIL_TIER2,Tier 1
5,RETENTION_VOICE_T2,Tier 3
8,APAC_CORE_EMAIL_T2,Tier 3
10,APAC_CHAT_SUPPORT_T1,Tier 2
12,AMER_CORE_EMAIL_TIER2,Tier 1
13,RETENTION_VOICE_T2,Tier 3
16,AMER_RETAIL_VOICE_T2,Tier 1
18,INBOUND_VOICE_TIER1,Tier 2


In [39]:
df_trabajo["validacion_tier"] = "No verificable"

df_trabajo.loc[
    tier_comparable & ~conflicto_tier,
    "validacion_tier"
] = "Coincide"

df_trabajo.loc[
    conflicto_tier,
    "validacion_tier"
] = "Diferencia"

print(df_trabajo["validacion_tier"].value_counts())

validacion_tier
Diferencia    6658
Coincide      3342
Name: count, dtype: int64


In [40]:
alertas = pd.DataFrame({
    "diferencia_canal": df_trabajo["validacion_canal"].eq("Conflicto"),
    "diferencia_tier": df_trabajo["validacion_tier"].eq("Diferencia"),
    "atencion_sin_agentes": df_trabajo["alerta_atencion_sin_agentes"]
})

df_trabajo["cantidad_alertas"] = alertas.sum(axis=1)

print("Conteo por tipo de alerta:")
print(alertas.sum())

print("\nDistribución de registros por cantidad de alertas:")
print(
    df_trabajo["cantidad_alertas"]
    .value_counts()
    .sort_index()
)

print(
    "\nRegistros con al menos una alerta:",
    df_trabajo["cantidad_alertas"].gt(0).sum()
)

Conteo por tipo de alerta:
diferencia_canal        5980
diferencia_tier         6658
atencion_sin_agentes     633
dtype: int64

Distribución de registros por cantidad de alertas:
cantidad_alertas
0    1220
1    4543
2    3983
3     254
Name: count, dtype: int64

Registros con al menos una alerta: 8780


In [41]:
meses = {
    "January": 1, "February": 2, "March": 3,
    "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9,
    "October": 10, "November": 11, "December": 12
}

mes_declarado = (
    df_trabajo["Month"]
    .str.strip()
    .str.title()
    .map(meses)
)

diferencia_anio = (
    df_trabajo["Year"]
    .ne(df_trabajo["Date"].dt.year)
)

mes_comparable = mes_declarado.notna()

diferencia_mes = (
    mes_comparable
    & mes_declarado.ne(df_trabajo["Date"].dt.month)
)

print("Diferencias de año:", diferencia_anio.sum())
print("Meses no reconocidos:", (~mes_comparable).sum())
print("Diferencias de mes:", diferencia_mes.sum())

Diferencias de año: 0
Meses no reconocidos: 0
Diferencias de mes: 0


In [42]:
hora_invalida = (
    df_trabajo["Hour"].isna()
    | ~df_trabajo["Hour"].between(0, 23)
    | df_trabajo["Hour"].mod(1).ne(0)
)

print("Horas inválidas:", hora_invalida.sum())

Horas inválidas: 0


In [43]:
cobertura_mensual = (
    df_trabajo
    .groupby(df_trabajo["Date"].dt.to_period("M"))
    .agg(
        registros=("Date", "size"),
        dias_con_registros=("Date", "nunique"),
        contactos_ofrecidos=("offered_Volume", "sum")
    )
)

# Incluir también meses completos sin observaciones
meses_del_periodo = pd.period_range(
    start=df_trabajo["Date"].min(),
    end=df_trabajo["Date"].max(),
    freq="M"
)

cobertura_mensual = cobertura_mensual.reindex(
    meses_del_periodo,
    fill_value=0
)

cobertura_mensual.index.name = "mes"

print(cobertura_mensual.to_string())

         registros  dias_con_registros  contactos_ofrecidos
mes                                                        
2021-01        181                  30                51676
2021-02        143                  28                38778
2021-03        168                  30                47235
2021-04        178                  30                49806
2021-05        190                  31                54385
2021-06        169                  29                49751
2021-07        147                  31                41150
2021-08        176                  31                50583
2021-09        168                  30                47651
2021-10        133                  30                35940
2021-11        148                  30                42984
2021-12        153                  31                39930
2022-01        168                  31                47203
2022-02        137                  28                39019
2022-03        184                  31  

In [44]:
dias_del_periodo = pd.date_range(
    start=df_trabajo["Date"].min(),
    end=df_trabajo["Date"].max(),
    freq="D"
)

dias_observados = pd.DatetimeIndex(
    df_trabajo["Date"].drop_duplicates()
)

dias_sin_registros = dias_del_periodo.difference(
    dias_observados
)

print("Días del período:", len(dias_del_periodo))
print("Días con registros:", len(dias_observados))
print("Días sin registros:", len(dias_sin_registros))

Días del período: 2163
Días con registros: 1885
Días sin registros: 278


In [45]:
fechas_2026 = (
    df_trabajo.loc[
        df_trabajo["Date"].dt.year.eq(2026),
        ["Date"]
    ]
    .drop_duplicates()
    .sort_values("Date")
)

dias_por_mes_2026 = (
    fechas_2026
    .groupby(fechas_2026["Date"].dt.to_period("M"))["Date"]
    .apply(lambda fechas: fechas.dt.day.tolist())
)

print(dias_por_mes_2026.to_string())

Date
2026-01    [1, 2, 3, 13, 14, 15, 16, 17, 18, 19, 20, 21, ...
2026-02    [1, 2, 3, 13, 14, 15, 16, 17, 18, 19, 20, 21, ...
2026-03            [1, 2, 3, 13, 14, 15, 16, 17, 18, 19, 20]
2026-04                                            [1, 2, 3]
2026-05                                            [1, 2, 3]
2026-06                                            [1, 2, 3]
2026-07                                            [1, 2, 3]
2026-08                                            [1, 2, 3]
2026-09                                            [1, 2, 3]
2026-10                                            [1, 2, 3]
2026-11                                            [1, 2, 3]
2026-12                                            [1, 2, 3]
Freq: M


In [46]:
dias_semana = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

dia_declarado = (
    df_trabajo["Weekday"]
    .str.strip()
    .str.title()
    .map(dias_semana)
)

dia_comparable = dia_declarado.notna()

diferencia_dia_semana = (
    dia_comparable
    & dia_declarado.ne(df_trabajo["Date"].dt.dayofweek)
)

print("Días de semana no reconocidos:", (~dia_comparable).sum())

resumen_dias = (
    pd.DataFrame({
        "anio": df_trabajo["Date"].dt.year,
        "comparable": dia_comparable,
        "diferencia": diferencia_dia_semana
    })
    .groupby("anio")
    .agg(
        registros=("comparable", "size"),
        comparables=("comparable", "sum"),
        diferencias=("diferencia", "sum")
    )
)

print(resumen_dias)

Días de semana no reconocidos: 0
      registros  comparables  diferencias
anio                                     
2021       1954         1954            0
2022       1893         1893            0
2023       2042         2042            0
2024       1806         1806            0
2025       1855         1855            0
2026        450          450            0


In [47]:
revision_dias = df_trabajo.loc[
    diferencia_dia_semana,
    ["Date", "Weekday", "Month", "Year"]
].copy()

revision_dias["dia_calculado"] = (
    revision_dias["Date"]
    .dt.dayofweek
    .map({numero: nombre for nombre, numero in dias_semana.items()})
)

print(revision_dias.head(10).to_string(index=False))

Empty DataFrame
Columns: [Date, Weekday, Month, Year, dia_calculado]
Index: []


In [48]:
resumen_calidad = pd.DataFrame([
    {
        "control": "Diferencia entre canal y nombre de cola",
        "registros": df_trabajo["validacion_canal"].eq("Conflicto").sum(),
        "tratamiento": "Conservar y señalar"
    },
    {
        "control": "Canal no verificable mediante nombre",
        "registros": df_trabajo["validacion_canal"].eq("No verificable").sum(),
        "tratamiento": "Conservar como no verificable"
    },
    {
        "control": "Diferencia de Tier",
        "registros": df_trabajo["validacion_tier"].eq("Diferencia").sum(),
        "tratamiento": "Conservar; confirmar significado"
    },
    {
        "control": "Atención con cero agentes",
        "registros": df_trabajo["alerta_atencion_sin_agentes"].sum(),
        "tratamiento": "Conservar; limitar métricas de personal"
    },
    {
        "control": "Backlog igual a abandonados",
        "registros": df_trabajo["backlog_Volume"]
            .eq(df_trabajo["abandoned_Volume"]).sum(),
        "tratamiento": "No usar como KPI independiente de backlog"
    }
])

resumen_calidad["porcentaje_registros"] = (
    resumen_calidad["registros"] / len(df_trabajo) * 100
).round(2)

resumen_calidad

,control,registros,tratamiento,porcentaje_registros
0,Diferencia entre canal y nombre de cola,5980,Conservar y señalar,59.80
1,Canal no verificable mediante nombre,986,Conservar como no verificable,9.86
2,Diferencia de Tier,6658,Conservar; confirmar significado,66.58
3,Atención con cero agentes,633,Conservar; limitar métricas de personal,6.33
4,Backlog igual a abandonados,10000,No usar como KPI independiente de backlog,100.00


In [49]:
df_preparado = df_trabajo.copy()

In [50]:
df_preparado.columns = (
    df_preparado.columns
    .str.strip()
    .str.lower()
)

In [51]:
for columna in df_preparado.columns:
    print(columna)

date
weekday
week_number
month
year
hour
country
region
site
queue_name
channel
all_units_filter
queue_units_filter
asu_tier_filter
business_unit
planned_volume
offered_volume
handled_volume
abandoned_volume
backlog_volume
aht_seconds
handle_time_total
wait_time_avg
shrinkage_percent
required_fte
agent_count
utilization_percent
active_agents
service_level_percent
occupancy_percent
validacion_canal
alerta_atencion_sin_agentes
validacion_tier
cantidad_alertas


In [52]:
print("Filas:", len(df_preparado))
print("Columnas:", len(df_preparado.columns))
print("Nombres únicos:", df_preparado.columns.is_unique)

Filas: 10000
Columnas: 34
Nombres únicos: True


In [53]:
tipos_actuales = (
    df_preparado.dtypes
    .astype(str)
    .rename("tipo_pandas")
    .to_frame()
)

print(tipos_actuales.to_string())

                                tipo_pandas
date                         datetime64[ns]
weekday                              object
week_number                           int64
month                                object
year                                  int64
hour                                  int64
country                              object
region                               object
site                                 object
queue_name                           object
channel                              object
all_units_filter                     object
queue_units_filter                   object
asu_tier_filter                      object
business_unit                        object
planned_volume                        int64
offered_volume                        int64
handled_volume                        int64
abandoned_volume                      int64
backlog_volume                        int64
aht_seconds                           int64
handle_time_total               

In [54]:
columnas_texto = df_preparado.select_dtypes(
    include=["object", "string"]
).columns

longitudes_texto = (
    df_preparado[columnas_texto]
    .apply(lambda columna: columna.astype("string").str.len().max())
    .sort_values(ascending=False)
)

print(longitudes_texto)

queue_name            24
site                  18
queue_units_filter    16
country               14
validacion_canal      14
all_units_filter      10
business_unit         10
validacion_tier       10
weekday                9
month                  9
asu_tier_filter        6
channel                5
region                 4
dtype: int64


In [55]:
columnas_numericas = df_preparado.select_dtypes(
    include=["number"]
).columns

rangos_numericos = (
    df_preparado[columnas_numericas]
    .agg(["min", "max"])
    .T
)

print(rangos_numericos.to_string())

                          min            max
week_number               1.0      53.000000
year                   2021.0    2026.000000
hour                      0.0      23.000000
planned_volume           50.0     500.000000
offered_volume           31.0     530.000000
handled_volume            3.0     521.000000
abandoned_volume          0.0      30.000000
backlog_volume            0.0      30.000000
aht_seconds             180.0     900.000000
handle_time_total       801.0  455875.000000
wait_time_avg             0.0     675.783784
shrinkage_percent        15.0      35.000000
required_fte              0.0      15.000000
agent_count               0.0      20.000000
utilization_percent      65.0      95.000000
active_agents             0.0      18.000000
service_level_percent    70.0      95.000000
occupancy_percent        60.0      90.000000
cantidad_alertas          0.0       3.000000


In [56]:
print(
    df_preparado["wait_time_avg"]
    .head(10)
    .to_string(index=False, float_format=lambda valor: f"{valor:.10f}")
)

 5.7723577240
78.9009901000
44.7594936700
 6.2876712330
44.4615384600
28.0000000000
31.6750524100
13.0000000000
41.7717391300
46.8396946600
